## LangChain

LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs).

* It provides tools for connecting LLMs to external data sources, like databases or APIs, and for creating chains of actions.
* This allows developers to build sophisticated applications that go beyond simple text generation.
* LangChain facilitates tasks like document summarization, question answering, and creating agents that can interact with the real world.
* By offering modular components and abstractions, it accelerates LLM app development, making complex workflows more manageable.

For more information, please refer to https://www.langchain.com/

## Retrieval Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by grounding them in external knowledge sources. Instead of relying solely on their pre-trained data, RAG allows LLMs to retrieve relevant information from a database or knowledge base in real-time.

Here's how it works: when a user poses a question, the system first retrieves relevant documents or passages from the external source. This retrieved information is then combined with the user's query and fed into the LLM, which generates a response based on both the retrieved knowledge and its pre-existing understanding.

RAG improves accuracy, reduces hallucinations, and enables LLMs to answer questions about information they weren't originally trained on.

 This is particularly useful for applications requiring up-to-date or domain-specific knowledge. RAG effectively bridges the gap between static LLMs and dynamic, real-world data.

## RAG Using LangChain

We will build a Retreival Augmented Generation Application using the following steps that we have discussed in the lesson.
* Step 1: Document Loading
* Step 2: Splitting Text into Chunks
* Step 3: Storage Text as Vectorstore
* Step 4: Query and Retreival text
* Step 5: Output answer with retreival text and LLM Augmented Generation

Dataset: encyclopedia of medicine PDF file

![RAG Workflow](https://miro.medium.com/v2/resize:fit:1200/format:webp/1*-TPXZmeTpI9hezbZqA4NpA.png)

##Load Libraries

In [ ]:
!pip install -U langchain langchain_community langchain_groq langchain_chroma langchain_huggingface langchain_text_splitters chromadb python-dotenv gdown pypdf sentence-transformers


Download the .env into the colab virtual drive

In [ ]:
import gdown
url = 'https://drive.google.com/file/d/1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm/view?usp=drive_link'
output_path = '.env'
gdown.download(url, output_path, quiet=False,fuzzy=True)


In [ ]:
from dotenv import load_dotenv
import os
# load .env file to environment
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

---
> NOTE: Please DO NOT use the Groq API Key outside of this workshop. Get a free key at https://console.groq.com/keys and store it in your `.env` file as `GROQ_API_KEY=...`
---

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.9)


#Test the LLM
print(llm.invoke([{'role':'user', 'content':'Which is the largest country by area in the world?'}]).content)

##Step 1: Document Loading

Create a directory name data. Copy the pdf document into the directory created. Filename: encyclopedia-of-medicine.pdf

In [ ]:
!mkdir data

Read the pdf doucment into multiple pages of text.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# Extract Data From the PDF File
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

extracted_data = load_pdf_file(data='./data/')

In [ ]:
print(type(extracted_data), len(extracted_data))   # Organize by the pages. Total pages: 68.

In [ ]:
extracted_data[3]

In [ ]:
extracted_data[-1]

##Step 2: Splitting Text into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)  # size by characters
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

In [ ]:
text_chunks[0]

In [ ]:
text_chunks[1]

In [ ]:
text_chunks[559]

In [ ]:
text_chunks[-1]

##Step 3: Storage Text as Vectorstore

Vector databases are specialized databases designed to efficiently store, manage, and query high-dimensional vector data.

These vectors represent data points in a multi-dimensional space, capturing the semantic meaning or features of items like text, images, or audio. Unlike traditional databases that rely on structured data and exact matches, vector databases excel at similarity searches.

![Vector Space](https://images.contentstack.io/v3/assets/blt7151619cb9560896/blt5c3b8fcafa132cf2/667daaeb82ce1d23f7312f32/lorbgyz9ffm8ui8jh-vector-database-search1.png)

> **Note:** Groq does not currently offer an embeddings API, so this notebook uses a free, local Hugging Face sentence-transformer model (`sentence-transformers/all-MiniLM-L6-v2`) for embeddings, while Groq is used for the fast LLM generation step.

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_texts([t.page_content for t in text_chunks],
                             embeddings,
                             collection_name="meddoc",
                             persist_directory="./meddoc_db")

##Step 4: Query and Retreival text

In [ ]:
question = "What are Acupressure?"
results = vectordb.similarity_search(question, k=3)
print(len(results))
print(results[0])

In [ ]:
question = "What are Acupressure?"
results = vectordb.similarity_search(question, k=5)
print(len(results))
print(results[-1])

In [ ]:
# Specifying top k
retriever = vectordb.as_retriever(search_kwargs={ "k" : 10})
print(retriever.invoke("What are Acupressure?")[0])

##Step 5: Output answer with retreival text and LLM Augmented Generation

Augmentation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

TEMPLATE = """\
You are medical assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

Generation

Finally, we are going to create a RAG Chain. For that, we are going to use LCEL (LangChain Expression Language) Runnable function.



In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

setup_and_retrieval = RunnableParallel({"question": RunnablePassthrough(),
                                        "context": retriever })
output_parser = StrOutputParser()
retrieval_chain = setup_and_retrieval | rag_prompt | llm | output_parser
retrieval_chain.invoke( "What is Acupressure?")

In [ ]:
retrieval_chain.invoke( "What is the COVID?")


In [ ]:
print(retrieval_chain.invoke( "What are the differences between Acupressure and Acupuncture?"))


In [ ]:
print(retrieval_chain.invoke("What is parkinson's disease?"))

## Execrise
Build an HR RAG (LangChain) from two PDFs

Goal — learners will build a Retrieval-Augmented Generation system that answers HR policy and job posting questions using the two PDFs as its knowledge base. The system should return helpful answers and cite where the information came from (document + page or chunk id).



Files placed in data/:

data/VirtuWorks_Expanded_HR_Policy_SG.pdf

data/VirtuWorks_Job_Postings_Text.pdf